In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import PowerTransformer
import h5py

warnings.filterwarnings('ignore')
# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


In [11]:
"""
_DE.mat是动态熵
DE是2betaband(low, high)*3channel(FzCzPz)*16subject*7session
(2, 3, 16, 7)
"""

'\n_DE.mat是动态熵\nDE是2betaband(low, high)*3channel(FzCzPz)*16subject*7session\n(2, 3, 16, 7)\n'

In [8]:
# 使用原始字符串避免路径转义问题
# file_path = r"E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\eeg比值&熵\熵\A_DE_landing_betaband.mat"
file_path = r"E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\eeg比值&熵\熵\A_DE_landing_betaband.mat"

# 打开 .mat 文件
with h5py.File(file_path, "r") as f:
    print("🔹 文件中所有顶层变量：")
    print(list(f.keys()))

    print("\n🔹 文件层级结构：")


    def print_tree(name, obj):
        print(name)


    f.visititems(print_tree)

    # 示例：如果文件中有变量 "data"，这样读取：
    # data = f["data"][:]
    # print("\n读取的 data：", data)


🔹 文件中所有顶层变量：
['A_temp_mean']

🔹 文件层级结构：
A_temp_mean


In [12]:
"""
文件列表：
A_DE_landing_betaband.mat

B_DE_landing_betaband.mat

文件名含义：
其中A代表alcohol组，B代表control组，DE代表动态熵，landing代表降落阶段，betaband代表beta波段

文件内容：
每个文件内容的形状：2beta波段(low, high)*3channel(FzCzPz)*16subject*7session
names_A = ["曾奇", "王榕榕", "徐梓军", "徐文婷", "达格吉", "李天琦", "王威", "张渊博", "黄博文",
           "朱祖恩", "刘子皓", "李嘉文", "徐子焰", "李国荣", "杨珊珊", "付瑞晗"]

names_B = ["杜佳鑫", "冯科嘉", "陈妍", "胡浩男", "王子铭", "程腾宇", "朱娇娇", "祖丽米热", "毛华灏", "郭浚杰", "周鑫颜",
           "肖雨晨", "冯晓娅", "刘勇杰", "祝鹏程", "董嘉乐"]
"""

'\n文件列表：\nA_DE_landing_betaband.mat\n\nB_DE_landing_betaband.mat\n\n文件名含义：\n其中A代表alcohol组，B代表control组，DE代表动态熵，landing代表降落阶段，betaband代表beta波段\n\n文件内容：\n每个文件内容的形状：2beta波段(low, high)*3channel(FzCzPz)*16subject*7session\nnames_A = ["曾奇", "王榕榕", "徐梓军", "徐文婷", "达格吉", "李天琦", "王威", "张渊博", "黄博文",\n           "朱祖恩", "刘子皓", "李嘉文", "徐子焰", "李国荣", "杨珊珊", "付瑞晗"]\n\nnames_B = ["杜佳鑫", "冯科嘉", "陈妍", "胡浩男", "王子铭", "程腾宇", "朱娇娇", "祖丽米热", "毛华灏", "郭浚杰", "周鑫颜",\n           "肖雨晨", "冯晓娅", "刘勇杰", "祝鹏程", "董嘉乐"]\n'

In [14]:
# 所有文件路径整理成字典
file_paths = {
    "A_temp_mean": r"E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\eeg比值&熵\熵\A_DE_landing_betaband.mat",
    "B_temp_mean": r"E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\eeg比值&熵\熵\B_DE_landing_betaband.mat"
}
# _ratio.mat是比值
# 自动循环读取
for key, path in file_paths.items():
    with h5py.File(path, "r") as f:
        # 形状：2beta波段(low, high)*3channel(FzCzPz)*16subject*7session
        dset = f[key]  # dataset 名与 key 一致
        print(f"{key} 形状： {dset.shape}")

A_temp_mean 形状： (2, 3, 16, 7)
B_temp_mean 形状： (2, 3, 16, 7)


In [35]:

# 文件路径
file_paths = {
    "A_temp_mean": r"E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\eeg比值&熵\熵\A_DE_landing_betaband.mat",
    "B_temp_mean": r"E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\eeg比值&熵\熵\B_DE_landing_betaband.mat",
}

# 受试者名字
names_A = ["曾奇", "王榕榕", "徐梓军", "徐文婷", "达格吉", "李天琦", "王威", "张渊博", "黄博文",
           "朱祖恩", "刘子皓", "李嘉文", "徐子焰", "李国荣", "杨珊珊", "付瑞晗"]

names_B = ["杜佳鑫", "冯科嘉", "陈妍", "胡浩男", "王子铭", "程腾宇", "朱娇娇", "祖丽米热", "毛华灏", "郭浚杰", "周鑫颜",
           "肖雨晨", "冯晓娅", "刘勇杰", "祝鹏程", "董嘉乐"]

beta_names = ["beta_low", "beta_high"]
channel_names = ["Fz", "Cz", "Pz"]

all_rows = []

# ========== 生成长格式 DataFrame ==========
for key, path in file_paths.items():
    group = "Alcohol" if key.startswith("A") else "Control"
    names_list = names_A if group == "Alcohol" else names_B

    with h5py.File(path, "r") as f:
        data = np.array(f[key])  # 形状：(2,3,16,7)

        for b in range(2):
            for c in range(3):
                for s in range(16):
                    for ses in range(7):
                        all_rows.append({
                            "组别": group,
                            "受试者": names_list[s],
                            "session": ses + 1,
                            "beta": beta_names[b],
                            "channel": channel_names[c],
                            "value": data[b, c, s, ses]
                        })

df_long = pd.DataFrame(all_rows)

# ========== 透视成宽格式 ==========
df_wide = df_long.pivot_table(
    index=["组别", "受试者", "session"],
    columns=["channel", "beta"],
    values="value"
)

# 扁平化列名，如 "Fz_beta_low"
df_wide.columns = [f"DE_{ch}_{bt}" for ch, bt in df_wide.columns]

# 恢复成普通列
df_wide = df_wide.reset_index()

df_wide

,组别,受试者,session,DE_Cz_beta_high,DE_Cz_beta_low,DE_Fz_beta_high,DE_Fz_beta_low,DE_Pz_beta_high,DE_Pz_beta_low
0,Alcohol,付瑞晗,1,0.205041,0.117051,0.025857,0.080689,0.311381,0.190822
1,Alcohol,付瑞晗,2,0.524562,0.530528,0.348305,0.276095,0.214957,0.148971
2,Alcohol,付瑞晗,3,-0.258908,-0.125163,0.260005,0.177508,0.259267,0.124151
3,Alcohol,付瑞晗,4,-0.159490,0.392523,0.287928,0.244415,0.335461,0.253055
4,Alcohol,付瑞晗,5,0.314410,0.028311,0.420035,0.136193,0.296648,0.206192
...,...,...,...,...,...,...,...,...,...
219,Control,陈妍,3,-0.332961,0.381392,0.300467,0.223729,0.435934,-0.002783
220,Control,陈妍,4,0.232564,0.143382,0.240019,-0.145777,0.335301,0.145758
221,Control,陈妍,5,0.605060,0.605060,0.605060,0.605060,0.605060,0.605060
222,Control,陈妍,6,-0.046973,0.166768,0.254342,0.268078,0.213953,0.045828


In [36]:
df_else = pd.read_excel(
    r"E:\pycharm all files\眼动数据处理\相关性分析\数据\预处理后的数据\按session分开的_eeg&eye&scl&ppg+eeg_ratio.xlsx")
df_else

,组别,受试者,session,SCL_mean_yj,AOI转换次数,静态注释熵(SGE),眼跳注视熵(GTE),HR,SDNN,RMSSD,...,Cz_theVShi,Cz_theVSlow,Fz_alpVShi,Fz_alpVSlow,Fz_theVShi,Fz_theVSlow,Pz_alpVShi,Pz_alpVSlow,Pz_theVShi,Pz_theVSlow
0,Alcohol,付瑞晗,1,0.008423,5.964286,0.793663,0.297438,17.819101,-3.260506,1.857342,...,-1.287953,-1.373074,-4.117628,-2.761845,-4.050715,-2.716964,1.460518,1.572004,1.308313,1.408181
1,Alcohol,付瑞晗,2,0.008640,10.500000,0.884336,0.313038,5.119280,-15.197881,-21.082419,...,1.666796,1.667920,1.186010,1.392895,0.911831,1.070889,2.053216,5.278246,1.767573,4.543939
2,Alcohol,付瑞晗,3,0.008858,15.035714,0.975010,0.328639,9.814463,11.623381,13.436624,...,-4.649697,1.317241,1.020330,1.174340,0.878509,1.011112,1.058269,0.795306,0.601766,0.452237
3,Alcohol,付瑞晗,4,0.009075,19.571429,1.065683,0.344240,44.991362,25.830811,31.173141,...,0.056982,0.116811,0.402521,1.264330,0.286452,0.899754,-5.516007,0.603882,-8.017515,0.877743
4,Alcohol,付瑞晗,5,0.009293,24.107143,1.156356,0.359840,10.378346,33.789581,41.231056,...,0.707111,0.724440,0.654791,0.786245,0.467395,0.561229,0.654377,1.452079,0.224516,0.498206
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,Control,陈妍,3,-0.070973,21.535714,1.521286,0.548103,-2.053871,38.025352,38.134196,...,4.543758,-1.649062,5.067337,59.993978,4.614078,54.627684,10.506800,8.394237,10.802331,8.630346
220,Control,陈妍,4,-0.088214,18.071429,1.326411,0.441623,-6.284982,-16.635726,-24.647037,...,2.634036,2.996113,-28.040737,21.678034,-23.036226,17.809093,-0.459137,1.321222,0.309093,-0.889454
221,Control,陈妍,5,-0.105455,14.607143,1.131536,0.335144,-1.159217,17.134339,17.823007,...,1.589667,-1.248641,-0.514815,-0.534875,0.899574,0.934625,-0.275669,-0.242793,1.219859,1.074378
222,Control,陈妍,6,-0.122697,11.142857,0.936661,0.228664,-5.534589,-22.075732,-34.098901,...,2.631512,6.660803,-3.974958,-1.820669,-3.265536,-1.495729,-1.902672,1.656347,1.280889,-1.115062


In [37]:
df_merged = pd.merge(
    df_else,
    df_wide,
    on=["组别", "受试者", "session"],
    how="inner"
)
df_merged

,组别,受试者,session,SCL_mean_yj,AOI转换次数,静态注释熵(SGE),眼跳注视熵(GTE),HR,SDNN,RMSSD,...,Pz_alpVShi,Pz_alpVSlow,Pz_theVShi,Pz_theVSlow,DE_Cz_beta_high,DE_Cz_beta_low,DE_Fz_beta_high,DE_Fz_beta_low,DE_Pz_beta_high,DE_Pz_beta_low
0,Alcohol,付瑞晗,1,0.008423,5.964286,0.793663,0.297438,17.819101,-3.260506,1.857342,...,1.460518,1.572004,1.308313,1.408181,0.205041,0.117051,0.025857,0.080689,0.311381,0.190822
1,Alcohol,付瑞晗,2,0.008640,10.500000,0.884336,0.313038,5.119280,-15.197881,-21.082419,...,2.053216,5.278246,1.767573,4.543939,0.524562,0.530528,0.348305,0.276095,0.214957,0.148971
2,Alcohol,付瑞晗,3,0.008858,15.035714,0.975010,0.328639,9.814463,11.623381,13.436624,...,1.058269,0.795306,0.601766,0.452237,-0.258908,-0.125163,0.260005,0.177508,0.259267,0.124151
3,Alcohol,付瑞晗,4,0.009075,19.571429,1.065683,0.344240,44.991362,25.830811,31.173141,...,-5.516007,0.603882,-8.017515,0.877743,-0.159490,0.392523,0.287928,0.244415,0.335461,0.253055
4,Alcohol,付瑞晗,5,0.009293,24.107143,1.156356,0.359840,10.378346,33.789581,41.231056,...,0.654377,1.452079,0.224516,0.498206,0.314410,0.028311,0.420035,0.136193,0.296648,0.206192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,Control,陈妍,3,-0.070973,21.535714,1.521286,0.548103,-2.053871,38.025352,38.134196,...,10.506800,8.394237,10.802331,8.630346,-0.332961,0.381392,0.300467,0.223729,0.435934,-0.002783
220,Control,陈妍,4,-0.088214,18.071429,1.326411,0.441623,-6.284982,-16.635726,-24.647037,...,-0.459137,1.321222,0.309093,-0.889454,0.232564,0.143382,0.240019,-0.145777,0.335301,0.145758
221,Control,陈妍,5,-0.105455,14.607143,1.131536,0.335144,-1.159217,17.134339,17.823007,...,-0.275669,-0.242793,1.219859,1.074378,0.605060,0.605060,0.605060,0.605060,0.605060,0.605060
222,Control,陈妍,6,-0.122697,11.142857,0.936661,0.228664,-5.534589,-22.075732,-34.098901,...,-1.902672,1.656347,1.280889,-1.115062,-0.046973,0.166768,0.254342,0.268078,0.213953,0.045828


In [38]:
df_merged.to_excel(
    "E:\pycharm all files\眼动数据处理\相关性分析\数据\预处理后的数据\按session分开的_eeg&eye&scl&ppg+eeg_ratio+eeg_de.xlsx",
    index=False)